# Trabajo Práctico: Búsqueda Voraz y A*

**Curso:** Inteligencia Artificial  
**Estudiante:** Dylan Gross  
**N° de alumno:** 19497  
**Fecha:** 08/09/2026  

---

## Introducción

En este TP vamos a implementar tres algoritmos de búsqueda sobre un grafo dirigido: **UCS** (costo uniforme), **Voraz** (el que siempre va al que parece más cerca) y **A*** (que combina los dos anteriores).

El objetivo es ver cómo se comporta cada uno, qué camino encuentra, cuánto cuesta y cuántos estados explora. La idea no es solo que funcionen, sino entender las diferencias entre ellos.

---

## El Grafo

Tenemos 6 estados: S, A, B, C, D, G.  
S es el inicio y G es el objetivo.

### Aristas (con sus costos):
- S → A (2)
- S → B (2)
- A → C (2)
- A → D (5)
- B → D (2)
- C → G (3)
- D → G (6)

### Heurística (estimación de cuánto falta para llegar a G):
| Estado | S | A | B | C | D | G |
|--------|---|---|---|---|---|---|
| h(n)   | 7 | 5 | 7 | 3 | 6 | 0 |

**Regla de desempate:** Si dos nodos tienen la misma prioridad, se saca el que se insertó primero (FIFO).

---

## Cómo funciona cada algoritmo

| Algoritmo | Usa para decidir | Qué busca |
|-----------|------------------|-----------|
| UCS       | g (costo recorrido) | El camino más barato |
| Voraz     | h (heurística) | El que parece más cerca |
| A*        | g + h | El mejor costo total estimado |



In [19]:
# Importamos las bibliotecas necesarias
import heapq
from collections import defaultdict

In [20]:
# Grafo dirigido con costos
grafo = {
    'S': {'A': 2, 'B': 2},
    'A': {'C': 2, 'D': 5},
    'B': {'D': 2},
    'C': {'G': 3},
    'D': {'G': 6},
    'G': {}
}

# Heurística (estimación del costo restante hasta G)
heuristica = {
    'S': 7,
    'A': 5,
    'B': 7,
    'C': 3,
    'D': 6,
    'G': 0
}

# Estado inicial y objetivo
INICIO = 'S'
OBJETIVO = 'G'

In [21]:
def crear_nodo(estado, padre=None, accion=None, g=0, h=0):
    """
    Crea un nodo para los algoritmos de búsqueda.

    Args:
        estado (str): El estado actual (ej: 'S', 'A', 'B')
        padre (dict): El nodo padre (None para la raíz)
        accion (str): La acción que generó este nodo (ej: 'S→A')
        g (float): Costo acumulado desde el inicio
        h (float): Heurística (estimación del costo restante)

    Returns:
        dict: Un nodo con toda la información
    """
    return {
        'estado': estado,
        'padre': padre,
        'accion': accion,
        'g': g,
        'h': h,
        'f': g + h  # Para A* (para UCS y Voraz se usa solo g o h)
    }

In [22]:
# Creamos el nodo raíz (S)
raiz = crear_nodo('S', h=heuristica['S'])
print("Nodo raíz:", raiz)

# Creamos el nodo A (hijo de S)
nodo_a = crear_nodo('A', padre=raiz, accion='S→A', g=2, h=heuristica['A'])
print("Nodo A:", nodo_a)

Nodo raíz: {'estado': 'S', 'padre': None, 'accion': None, 'g': 0, 'h': 7, 'f': 7}
Nodo A: {'estado': 'A', 'padre': {'estado': 'S', 'padre': None, 'accion': None, 'g': 0, 'h': 7, 'f': 7}, 'accion': 'S→A', 'g': 2, 'h': 5, 'f': 7}


In [23]:
# Creamos los nodos necesarios

# Nodo raíz (S)
raiz = crear_nodo('S', h=heuristica['S'])

# Nodo A (hijo de S)
nodo_a = crear_nodo('A', padre=raiz, accion='S->A', g=2, h=heuristica['A'])

# Nodo C (hijo de A)
nodo_c = crear_nodo('C', padre=nodo_a, accion='A->C', g=4, h=heuristica['C'])

# Nodo G (hijo de C)
nodo_g = crear_nodo('G', padre=nodo_c, accion='C->G', g=7, h=heuristica['G'])

camino, acciones, costo = reconstruir_camino(nodo_g)

print("Camino:", camino)
print("Acciones:", acciones)
print("Costo total:", costo)

Camino: ['S', 'A', 'C', 'G']
Acciones: ['S->A', 'A->C', 'C->G']
Costo total: 7


In [24]:
def generar_sucesores(nodo, grafo, heuristica):
    """
    Genera todos los sucesores válidos de un nodo.

    Args:
        nodo (dict): El nodo actual
        grafo (dict): El grafo con las transiciones
        heuristica (dict): La heurística de cada estado

    Returns:
        list: Lista de nodos sucesores
    """
    sucesores = []
    estado_actual = nodo['estado']

    # Si el estado no tiene transiciones, devolvemos lista vacía
    if estado_actual not in grafo:
        return sucesores

    # Recorremos cada vecino del estado actual
    for estado_siguiente, costo_arista in grafo[estado_actual].items():
        # Calculamos el nuevo costo acumulado
        nuevo_g = nodo['g'] + costo_arista

        # Creamos el nodo sucesor
        sucesor = crear_nodo(
            estado=estado_siguiente,
            padre=nodo,
            accion=f"{estado_actual}->{estado_siguiente}",
            g=nuevo_g,
            h=heuristica[estado_siguiente]
        )
        sucesores.append(sucesor)

    return sucesores

In [25]:
# Generamos los sucesores de la raíz (S)
sucesores = generar_sucesores(raiz, grafo, heuristica)

print("Sucesores de S:")
for s in sucesores:
    print(f"  Estado: {s['estado']}, g={s['g']}, h={s['h']}, f={s['f']}, accion={s['accion']}")

Sucesores de S:
  Estado: A, g=2, h=5, f=7, accion=S->A
  Estado: B, g=2, h=7, f=9, accion=S->B


In [26]:
def busqueda(grafo, heuristica, inicio, objetivo, tipo):
    """
    Algoritmo de búsqueda genérico (UCS, Voraz o A*).

    Args:
        grafo (dict): El grafo con las transiciones
        heuristica (dict): La heurística de cada estado
        inicio (str): Estado inicial
        objetivo (str): Estado objetivo
        tipo (str): 'ucs', 'voraz' o 'a_estrella'

    Returns:
        dict: Resultados con camino, costo, estadísticas y traza
    """
    # Diccionario para guardar el mejor costo conocido de cada estado
    mejor_g = {}

    # Estadísticas
    stats = {
        'generados': 0,
        'expandidos': 0,
        'frontera_maxima': 0,
        'reaperturas': 0
    }

    # Traza de lo que va pasando
    traza = []

    # Contador para el desempate FIFO
    contador = 0

    # Definimos la prioridad según el tipo de búsqueda
    if tipo == 'ucs':
        prioridad = lambda n: n['g']
    elif tipo == 'voraz':
        prioridad = lambda n: n['h']
    elif tipo == 'a_estrella':
        prioridad = lambda n: n['f']
    else:
        raise ValueError("tipo debe ser 'ucs', 'voraz' o 'a_estrella'")

    # Creamos el nodo inicial
    nodo_inicial = crear_nodo(inicio, h=heuristica[inicio])
    mejor_g[inicio] = 0
    stats['generados'] += 1

    # Frontera (cola de prioridad)
    frontera = []
    heapq.heappush(frontera, (prioridad(nodo_inicial), contador, nodo_inicial))
    contador += 1
    stats['frontera_maxima'] = len(frontera)

    # Registramos el inicio en la traza
    traza.append({
        'accion': 'INICIO',
        'estado': inicio,
        'frontera': [(prioridad(n), n['estado']) for _, _, n in frontera]
    })

    # Bucle principal
    while frontera:
        # Sacamos el nodo con menor prioridad
        _, _, nodo_actual = heapq.heappop(frontera)
        stats['expandidos'] += 1

        # Si es obsoleto, lo ignoramos
        if nodo_actual['g'] > mejor_g.get(nodo_actual['estado'], float('inf')):
            continue

        # Generamos los sucesores
        sucesores = generar_sucesores(nodo_actual, grafo, heuristica)
        stats['generados'] += len(sucesores)

        # Relajamos cada sucesor
        for sucesor in sucesores:
            estado = sucesor['estado']
            nuevo_g = sucesor['g']

            if nuevo_g < mejor_g.get(estado, float('inf')):
                if estado in mejor_g:
                    stats['reaperturas'] += 1
                mejor_g[estado] = nuevo_g
                heapq.heappush(frontera, (prioridad(sucesor), contador, sucesor))
                contador += 1
                stats['frontera_maxima'] = max(stats['frontera_maxima'], len(frontera))

        # Registramos la expansión
        traza.append({
            'accion': 'EXPANDIR',
            'estado': nodo_actual['estado'],
            'g': nodo_actual['g'],
            'h': nodo_actual['h'],
            'f': nodo_actual['f'],
            'frontera': [(prioridad(n), n['estado']) for _, _, n in frontera]
        })

        # Si es el objetivo, devolvemos el camino
        if nodo_actual['estado'] == objetivo:
            camino, acciones, costo = reconstruir_camino(nodo_actual)
            return {
                'camino': camino,
                'acciones': acciones,
                'costo': costo,
                'expandidos': stats['expandidos'],
                'generados': stats['generados'],
                'frontera_maxima': stats['frontera_maxima'],
                'reaperturas': stats['reaperturas'],
                'traza': traza
            }

    return None

In [27]:
resultado = busqueda(grafo, heuristica, 'S', 'G', 'ucs')

print("=== UCS ===")
print("Camino:", resultado['camino'])
print("Costo:", resultado['costo'])
print("Expandidos:", resultado['expandidos'])
print("Generados:", resultado['generados'])
print("Reaperturas:", resultado['reaperturas'])
print("Frontera máxima:", resultado['frontera_maxima'])

=== UCS ===
Camino: ['S', 'A', 'C', 'G']
Costo: 7
Expandidos: 7
Generados: 8
Reaperturas: 1
Frontera máxima: 3


In [28]:
# Probamos Voraz
resultado_voraz = busqueda(grafo, heuristica, 'S', 'G', 'voraz')

print("=== Voraz ===")
print("Camino:", resultado_voraz['camino'])
print("Costo:", resultado_voraz['costo'])
print("Expandidos:", resultado_voraz['expandidos'])
print("Generados:", resultado_voraz['generados'])
print("Reaperturas:", resultado_voraz['reaperturas'])
print("Frontera máxima:", resultado_voraz['frontera_maxima'])

print()

# Probamos A*
resultado_a = busqueda(grafo, heuristica, 'S', 'G', 'a_estrella')

print("=== A* ===")
print("Camino:", resultado_a['camino'])
print("Costo:", resultado_a['costo'])
print("Expandidos:", resultado_a['expandidos'])
print("Generados:", resultado_a['generados'])
print("Reaperturas:", resultado_a['reaperturas'])
print("Frontera máxima:", resultado_a['frontera_maxima'])

=== Voraz ===
Camino: ['S', 'A', 'C', 'G']
Costo: 7
Expandidos: 4
Generados: 6
Reaperturas: 0
Frontera máxima: 3

=== A* ===
Camino: ['S', 'A', 'C', 'G']
Costo: 7
Expandidos: 4
Generados: 6
Reaperturas: 0
Frontera máxima: 3


In [29]:
# Guardamos los resultados en un diccionario
resultados = {
    'UCS': busqueda(grafo, heuristica, 'S', 'G', 'ucs'),
    'Voraz': busqueda(grafo, heuristica, 'S', 'G', 'voraz'),
    'A*': busqueda(grafo, heuristica, 'S', 'G', 'a_estrella')
}

# Mostramos la tabla comparativa
print("="*70)
print("TABLA COMPARATIVA")
print("="*70)
print(f"{'Algoritmo':<10} {'Camino':<25} {'Costo':<8} {'Prioridad':<10} {'Expandidos':<10}")
print("-"*70)

for nombre, res in resultados.items():
    camino_str = " → ".join(res['camino'])
    prioridad = {'UCS': 'g', 'Voraz': 'h', 'A*': 'g+h'}[nombre]
    print(f"{nombre:<10} {camino_str:<25} {res['costo']:<8} {prioridad:<10} {res['expandidos']:<10}")

print("="*70)

TABLA COMPARATIVA
Algoritmo  Camino                    Costo    Prioridad  Expandidos
----------------------------------------------------------------------
UCS        S → A → C → G             7        g          7         
Voraz      S → A → C → G             7        h          4         
A*         S → A → C → G             7        g+h        4         


In [30]:
def mostrar_traza(resultado, nombre):
    print(f"\n{'='*70}")
    print(f"TRAZA DE {nombre}")
    print(f"{'='*70}")
    for i, paso in enumerate(resultado['traza']):
        if paso['accion'] == 'INICIO':
            print(f"Paso {i}: INICIO")
            print(f"   Frontera: {paso['frontera']}")
        else:
            print(f"Paso {i}: EXPANDIR {paso['estado']} (g={paso['g']}, h={paso['h']}, f={paso['f']})")
            print(f"   Frontera: {paso['frontera']}")

# Mostramos las trazas
mostrar_traza(resultados['UCS'], 'UCS')
mostrar_traza(resultados['Voraz'], 'Voraz')
mostrar_traza(resultados['A*'], 'A*')


TRAZA DE UCS
Paso 0: INICIO
   Frontera: [(0, 'S')]
Paso 1: EXPANDIR S (g=0, h=7, f=7)
   Frontera: [(2, 'A'), (2, 'B')]
Paso 2: EXPANDIR A (g=2, h=5, f=7)
   Frontera: [(2, 'B'), (4, 'C'), (7, 'D')]
Paso 3: EXPANDIR B (g=2, h=7, f=9)
   Frontera: [(4, 'C'), (7, 'D'), (4, 'D')]
Paso 4: EXPANDIR C (g=4, h=3, f=7)
   Frontera: [(4, 'D'), (7, 'D'), (7, 'G')]
Paso 5: EXPANDIR D (g=4, h=6, f=10)
   Frontera: [(7, 'D'), (7, 'G')]
Paso 6: EXPANDIR G (g=7, h=0, f=7)
   Frontera: []

TRAZA DE Voraz
Paso 0: INICIO
   Frontera: [(7, 'S')]
Paso 1: EXPANDIR S (g=0, h=7, f=7)
   Frontera: [(5, 'A'), (7, 'B')]
Paso 2: EXPANDIR A (g=2, h=5, f=7)
   Frontera: [(3, 'C'), (7, 'B'), (6, 'D')]
Paso 3: EXPANDIR C (g=4, h=3, f=7)
   Frontera: [(0, 'G'), (7, 'B'), (6, 'D')]
Paso 4: EXPANDIR G (g=7, h=0, f=7)
   Frontera: [(6, 'D'), (7, 'B')]

TRAZA DE A*
Paso 0: INICIO
   Frontera: [(7, 'S')]
Paso 1: EXPANDIR S (g=0, h=7, f=7)
   Frontera: [(7, 'A'), (9, 'B')]
Paso 2: EXPANDIR A (g=2, h=5, f=7)
   Frontera: 

## Análisis de Resultados

### 1. ¿Por qué voraz y A* coinciden en este grafo? ¿Qué condición del grafo y de la heurística lo explica?

Voraz y A* coinciden porque en este grafo la heurística está diseñada de forma tal que **el nodo que parece más cerca del objetivo (menor h) también es el que tiene el mejor costo total estimado (menor f = g+h)**.

Además, la heurística es **admisible** (nunca sobreestima el costo real) y **consistente** (se cumple que h(n) ≤ costo(n,n') + h(n') para toda arista). Esto hace que A* encuentre el camino óptimo sin necesidad de reabrir nodos.

En este grafo en particular, el camino óptimo (S → A → C → G) tiene valores de heurística que van bajando (7, 5, 3, 0), mientras que los caminos alternativos (pasando por B o D) tienen valores más altos en los puntos de decisión. Por eso ambos algoritmos eligen el mismo camino.

---

### 2. ¿Garantiza voraz devolver el camino de menor costo en general? Justificá con la propiedad de su prioridad.

**No, voraz NO garantiza el camino de menor costo.**

Su prioridad es **h(n)**, que es solo una estimación de lo que falta para llegar al objetivo. Voraz ignora por completo el costo ya recorrido (g). Esto significa que puede ser "engañado" por un estado que parece muy cerca del objetivo (h bajo) pero al que se llega por un camino muy caro.

**Ejemplo:** si hay un camino que cuesta 100 y tiene h=1, y otro que cuesta 5 y tiene h=2, voraz elegiría el primero (h=1) aunque el costo total sea mucho mayor.

Por eso voraz es **rápido pero no óptimo**: puede encontrar una solución, pero no necesariamente la más barata.

---

### 3. ¿Qué ocurre si se usa h = 0 en A*? ¿Con qué algoritmo coincide entonces?

Si usamos **h = 0** en A*, entonces la prioridad queda:

**f(n) = g(n) + 0 = g(n)**

Es decir, A* pasa a ordenar la frontera por **g** nada más. Eso es exactamente lo que hace **UCS (Costo Uniforme)**.

Por lo tanto, **A* con h=0 coincide con UCS**.

Esto tiene sentido porque A* es una generalización de UCS: cuando la heurística no aporta información (h=0), A* se comporta como una búsqueda no informada que solo se guía por el costo acumulado.

---

### 4. ¿Hubo reaperturas en UCS? ¿Y en A* y voraz? ¿Por qué?

- **UCS:** Sí, hubo **1 reapertura**. El estado D fue alcanzado primero desde A con g=7, y después desde B con g=4. Como 4 < 7, se actualizó mejor_g[D] y se reinsertó D con el nuevo costo. Eso cuenta como reapertura.
- **Voraz:** No hubo reaperturas. Como voraz no usa g para decidir, no le importa si encuentra un camino más barato después; simplemente sigue la heurística.
- **A*:** No hubo reaperturas. Gracias a que la heurística es **consistente**, A* nunca necesita reabrir un nodo ya expandido, porque el primer camino que encuentra a cada estado ya es el óptimo.

**¿Por qué pasan las reaperturas?**  
Una reapertura ocurre cuando encontramos un camino mejor (menor g) a un estado que ya habíamos visto antes. En UCS esto puede pasar porque exploramos en orden de costo y un estado puede alcanzarse por varios caminos. En A* con heurística consistente, esto no pasa porque la primera vez que se extrae un nodo, ya tiene el costo óptimo.

---

### 5. ¿"Expandir menos estados" significa "camino más barato"? Relacionalo con lo que muestran UCS y voraz aquí.

**No, expandir menos estados NO significa que el camino sea más barato.**

En este grafo:
- **UCS** expandió **6 estados** (S, A, B, C, D, G) y encontró el camino S→A→C→G con costo 7.
- **Voraz** expandió **4 estados** (S, A, C, G) y encontró el mismo camino con el mismo costo 7.

Voraz expandió menos porque la heurística lo guió directo hacia la solución, sin perder tiempo explorando B y D. Pero eso no significa que voraz sea "mejor" en general: en otros grafos, voraz podría expandir pocos estados y encontrar un camino más caro que el óptimo.

**Conclusión:** la cantidad de estados expandidos mide la **eficiencia** del algoritmo (cuánto tarda en encontrar una solución), no la **calidad** de la solución (qué tan barato es el camino). UCS garantiza optimalidad aunque expanda más; voraz es más rápido pero no garantiza nada.

---

## Tabla Comparativa Final

| Resultado | UCS | Voraz | A* |
|-----------|-----|-------|-----|
| **Camino** | S → A → C → G | S → A → C → G | S → A → C → G |
| **Costo** | 7 | 7 | 7 |
| **Prioridad** | g | h | g+h |
| **Expandidos antes de extraer G** | 6 | 4 | 4 |
| **Reaperturas** | 1 | 0 | 0 |
| **Frontera máxima** | 3 | 3 | 3 |

> **Nota:** En el código, el contador de "expandidos" incrementa también cuando se extrae un nodo obsoleto. Si no contamos los obsoletos, UCS expande 6 estados. Si los contamos, expande 7. En la tabla usamos el criterio conceptual (sin contar obsoletos).